# Explore PROMISE12 CSV manifests

Goal: collect the CSV manifests shipped in the Zenodo PROMISE12 download
(`training_data.csv`, `test_data.csv`, `livechallenge_test_data.csv`), and for each
row **actually open the referenced `t2` NIfTI file** to read its real header metadata
(size, spacing, origin, direction, pixel type) rather than just listing manifest paths.

The three CSVs share identical columns and overlapping `ID` numbering, so each row is
tagged with a `source` column (the originating CSV) before combining them, to keep
provenance clear.

In [1]:
from pathlib import Path

import pandas as pd
import SimpleITK as sitk

DATA_DIR = Path("../data/raw/zenodo/promise12/record-8026660/promise12")
assert DATA_DIR.exists(), DATA_DIR

In [2]:
def read_nii_header(path: Path) -> dict:
    img = sitk.ReadImage(str(path))
    size = img.GetSize()
    spacing = img.GetSpacing()
    origin = img.GetOrigin()
    return {
        "dim_x": size[0],
        "dim_y": size[1],
        "dim_z": size[2],
        "spacing_x": round(spacing[0], 4),
        "spacing_y": round(spacing[1], 4),
        "spacing_z": round(spacing[2], 4),
        "origin_x": round(origin[0], 2),
        "origin_y": round(origin[1], 2),
        "origin_z": round(origin[2], 2),
        "pixel_type": img.GetPixelIDTypeAsString(),
    }

In [3]:
csv_names = ["training_data.csv", "test_data.csv", "livechallenge_test_data.csv"]

dfs = []
for csv_name in csv_names:
    source = Path(csv_name).stem
    csv_df = pd.read_csv(DATA_DIR / csv_name)
    csv_df["source"] = source

    header_rows = [read_nii_header(DATA_DIR / t2_path) for t2_path in csv_df["t2"]]
    csv_df = pd.concat([csv_df, pd.DataFrame(header_rows)], axis=1)
    dfs.append(csv_df)

In [4]:
df_all = pd.concat(dfs, ignore_index=True)
df_all = df_all[
    [
        "source",
        "ID",
        "t2",
        "segmentation",
        "dim_x",
        "dim_y",
        "dim_z",
        "spacing_x",
        "spacing_y",
        "spacing_z",
        "origin_x",
        "origin_y",
        "origin_z",
        "pixel_type",
    ]
]
df_all

,source,ID,t2,segmentation,dim_x,dim_y,dim_z,spacing_x,spacing_y,spacing_z,origin_x,origin_y,origin_z,pixel_type
0,training_data,0,training_data_nii/Case00.nii.gz,training_data_nii/Case00_segmentation.nii.gz,512,512,47,0.2734,0.2734,2.2000,-71.99,-53.31,-55.75,16-bit signed integer
1,training_data,1,training_data_nii/Case01.nii.gz,training_data_nii/Case01_segmentation.nii.gz,512,512,29,0.3516,0.3516,3.0003,-93.04,-64.04,-138.27,16-bit signed integer
2,training_data,2,training_data_nii/Case02.nii.gz,training_data_nii/Case02_segmentation.nii.gz,512,512,54,0.2734,0.2734,2.2000,-68.25,-75.34,-37.34,16-bit signed integer
3,training_data,3,training_data_nii/Case03.nii.gz,training_data_nii/Case03_segmentation.nii.gz,512,512,42,0.2734,0.2734,2.2000,-88.90,-55.24,0.06,16-bit signed integer
4,training_data,4,training_data_nii/Case04.nii.gz,training_data_nii/Case04_segmentation.nii.gz,512,512,46,0.2734,0.2734,2.2005,-70.51,-63.28,-120.97,16-bit signed integer
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,livechallenge_test_data,15,livechallenge_test_data_nii/Case15.nii.gz,livechallenge_test_data_nii/Case15_segmentatio...,320,320,20,0.6250,0.6250,3.6000,-91.50,-93.19,-89.08,16-bit signed integer
96,livechallenge_test_data,16,livechallenge_test_data_nii/Case16.nii.gz,livechallenge_test_data_nii/Case16_segmentatio...,320,320,20,0.6250,0.6250,3.6000,-112.81,-100.82,-95.86,16-bit signed integer
97,livechallenge_test_data,17,livechallenge_test_data_nii/Case17.nii.gz,livechallenge_test_data_nii/Case17_segmentatio...,320,320,20,0.6250,0.6250,3.6000,-90.65,-96.63,-39.43,16-bit signed integer
98,livechallenge_test_data,18,livechallenge_test_data_nii/Case18.nii.gz,livechallenge_test_data_nii/Case18_segmentatio...,320,320,20,0.6250,0.6250,3.6000,-104.14,-91.61,-77.78,16-bit signed integer


In [5]:
print("Rows per source:")
print(df_all["source"].value_counts())

pd.set_option("display.max_rows", None)
df_all

Rows per source:
source
training_data              50
test_data                  30
livechallenge_test_data    20
Name: count, dtype: int64


,source,ID,t2,segmentation,dim_x,dim_y,dim_z,spacing_x,spacing_y,spacing_z,origin_x,origin_y,origin_z,pixel_type
0,training_data,0,training_data_nii/Case00.nii.gz,training_data_nii/Case00_segmentation.nii.gz,512,512,47,0.2734,0.2734,2.2000,-71.99,-53.31,-55.75,16-bit signed integer
1,training_data,1,training_data_nii/Case01.nii.gz,training_data_nii/Case01_segmentation.nii.gz,512,512,29,0.3516,0.3516,3.0003,-93.04,-64.04,-138.27,16-bit signed integer
2,training_data,2,training_data_nii/Case02.nii.gz,training_data_nii/Case02_segmentation.nii.gz,512,512,54,0.2734,0.2734,2.2000,-68.25,-75.34,-37.34,16-bit signed integer
3,training_data,3,training_data_nii/Case03.nii.gz,training_data_nii/Case03_segmentation.nii.gz,512,512,42,0.2734,0.2734,2.2000,-88.90,-55.24,0.06,16-bit signed integer
4,training_data,4,training_data_nii/Case04.nii.gz,training_data_nii/Case04_segmentation.nii.gz,512,512,46,0.2734,0.2734,2.2005,-70.51,-63.28,-120.97,16-bit signed integer
5,training_data,5,training_data_nii/Case05.nii.gz,training_data_nii/Case05_segmentation.nii.gz,512,512,42,0.2734,0.2734,2.2001,-67.25,-64.53,-61.88,16-bit signed integer
6,training_data,6,training_data_nii/Case06.nii.gz,training_data_nii/Case06_segmentation.nii.gz,512,512,39,0.3125,0.3125,2.7999,-87.87,-77.14,-48.04,16-bit signed integer
7,training_data,7,training_data_nii/Case07.nii.gz,training_data_nii/Case07_segmentation.nii.gz,512,512,34,0.2734,0.2734,2.9999,-53.96,-37.86,-80.35,16-bit signed integer
8,training_data,8,training_data_nii/Case08.nii.gz,training_data_nii/Case08_segmentation.nii.gz,512,512,40,0.3125,0.3125,2.8000,-94.79,-48.87,-63.69,16-bit signed integer
9,training_data,9,training_data_nii/Case09.nii.gz,training_data_nii/Case09_segmentation.nii.gz,512,512,47,0.2734,0.2734,2.2000,-63.72,-67.01,-66.14,16-bit signed integer


In [11]:
runmc_mask = df_all.eval("3.6 <= spacing_z & ~(dim_x == 320 & dim_y == 320 & dim_z == 24) & ~(origin_x == 0 & origin_y == 0 & origin_z == 0)")
df_all[runmc_mask]

,source,ID,t2,segmentation,dim_x,dim_y,dim_z,spacing_x,spacing_y,spacing_z,origin_x,origin_y,origin_z,pixel_type
13,training_data,13,training_data_nii/Case13.nii.gz,training_data_nii/Case13_segmentation.nii.gz,320,320,15,0.600,0.600,4.0,-93.23,-77.76,-94.50,16-bit signed integer
14,training_data,14,training_data_nii/Case14.nii.gz,training_data_nii/Case14_segmentation.nii.gz,320,320,20,0.625,0.625,3.6,-105.11,-92.74,-42.85,16-bit signed integer
15,training_data,15,training_data_nii/Case15.nii.gz,training_data_nii/Case15_segmentation.nii.gz,320,320,20,0.625,0.625,3.6,-113.17,-93.24,-54.66,16-bit signed integer
16,training_data,16,training_data_nii/Case16.nii.gz,training_data_nii/Case16_segmentation.nii.gz,320,320,20,0.625,0.625,3.6,-111.84,-115.22,-31.69,16-bit signed integer
17,training_data,17,training_data_nii/Case17.nii.gz,training_data_nii/Case17_segmentation.nii.gz,320,320,20,0.625,0.625,3.6,-92.47,-111.83,-57.84,16-bit signed integer
18,training_data,18,training_data_nii/Case18.nii.gz,training_data_nii/Case18_segmentation.nii.gz,320,320,17,0.600,0.600,4.0,-91.30,-96.11,-87.28,16-bit signed integer
19,training_data,19,training_data_nii/Case19.nii.gz,training_data_nii/Case19_segmentation.nii.gz,320,320,20,0.625,0.625,3.6,-84.86,-92.38,-63.49,16-bit signed integer
20,training_data,20,training_data_nii/Case20.nii.gz,training_data_nii/Case20_segmentation.nii.gz,320,320,20,0.625,0.625,3.6,-111.85,-96.57,-66.54,16-bit signed integer
21,training_data,21,training_data_nii/Case21.nii.gz,training_data_nii/Case21_segmentation.nii.gz,320,320,20,0.625,0.625,3.6,-92.47,-91.62,-0.58,16-bit signed integer
22,training_data,22,training_data_nii/Case22.nii.gz,training_data_nii/Case22_segmentation.nii.gz,320,320,20,0.625,0.625,3.6,-79.35,-91.39,-86.53,16-bit signed integer


In [14]:
# Write the pooled runmc data and non-runmc databack to csv
# Columns:
# ID - concatenate source and ID with underscore
# t2
# segmentation

def build_pooled_csv(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["ID"] = out["source"] + "_" + out["ID"].apply(lambda x: f"{x:02d}")
    return out[["ID", "t2", "segmentation"]]

runmc_df = build_pooled_csv(df_all[runmc_mask])
non_runmc_df = build_pooled_csv(df_all[~runmc_mask])

runmc_df.to_csv(DATA_DIR / "pooled_runmc.csv", index=False)
non_runmc_df.to_csv(DATA_DIR / "pooled_nonrunmc.csv", index=False)
